# 02477 Bayesian Machine Learning — Exam Notebook

**PDF export — run in terminal from the `Exam Prep/` directory:**
```bash
jupyter nbconvert --to html exam.ipynb
```
Then open **exam.html** in your browser and use **File → Print → Save as PDF** (Cmd+P on Mac).  


In [ ]:
# ── Master import — run this first ──────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'helpers'))
from helpers import *

import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

try:
    import jax
    import jax.numpy as jnp
    key = jax.random.PRNGKey(0)
    print('JAX available')
except ImportError:
    print('JAX not available')

print('All imports OK')

---
# Data Loading

In [ ]:
# .npz file
# data = np.load('data.npz')
# X_train, y_train = data['X_train'], data['y_train']
# X_test,  y_test  = data['X_test'],  data['y_test']

# .csv file
# import pandas as pd
# df = pd.read_csv('data.csv')
# X = df[['col1','col2']].values;  y = df['target'].values

# Design matrix (week 3 style)
# basis_fns = [lambda x: np.ones_like(x), lambda x: x, lambda x: x**2]
# Phi = np.column_stack([f(X_train.ravel()) for f in basis_fns])

---
# Week 1 — Beta-Binomial (Conjugate Model)

Prior: θ ~ Beta(a, b) | Likelihood: y|θ,N ~ Bin(N,θ) | Posterior: θ|y ~ Beta(a+y, b+N-y)

In [ ]:
prior = BetaDistribution(a=1, b=1)           # uniform prior
data  = ObservedData(y=7, N=10)
post  = compute_posterior(prior, data)

print(f'Posterior: Beta({post.a}, {post.b})')
print(f'Mean={post.mean:.4f}  Var={post.variance:.4f}  Mode={post.mode:.4f}')
print(f'95% CI: {post.get_interval(0.95)}')

fig, ax = plt.subplots(figsize=(6,3))
plot_beta(ax, prior, label='Prior')
plot_beta(ax, post,  label='Posterior')
ax.legend(); ax.set_xlabel(r'$\theta$'); plt.tight_layout(); plt.show()

log_ev = beta_binomial_log_evidence(y=7, N=10, a=1, b=1)
print(f'Log evidence: {log_ev:.4f}')

---
# Week 2 — Grid Approximation & Logistic Regression

σ(a) = 1/(1+exp(-a)) | log p(w|y) ∝ Σ y_n log σ(w^Tx_n) + (1-y_n)log(1-σ) - α/2 ||w||²

In [ ]:
# x, y, N = feature vector (1-D), binary targets, trials per obs
# model = LogisticRegression(x, y, N=1, sigma2_alpha=1.0, sigma2_beta=1.0)

# ── Grid approximation over (alpha, beta) ────────────────────────────────────
# import jax.numpy as jnp
# alphas = jnp.linspace(-4, 4, 90)
# betas  = jnp.linspace(-4, 4, 100)
# grid   = GridApproximation2D(alphas, betas, model.log_joint)
# grid.marginal_alpha.print_summary()   # mean, std, 95% CI for alpha
# grid.marginal_beta.print_summary()    # mean, std, 95% CI for beta

# ── Posterior predictive ─────────────────────────────────────────────────────
# x_new = 31.0   # value of feature
# p_hat = grid.compute_expectation(lambda a, b: sigmoid(a + b*x_new))
# print(f'P(y=1 | x={x_new}) ≈ {float(p_hat):.4f}')

# ── Sample from posterior ────────────────────────────────────────────────────
# key = jax.random.PRNGKey(0)
# a_samp, b_samp = grid.sample(key, num_samples=500)


---
# Week 3 — Bayesian Linear Regression

S_N = (αI + βΦ^TΦ)^{-1}  |  m_N = β S_N Φ^T y  |  p(y*|x*) = N(m_N^T φ*, β^{-1} + φ*^T S_N φ*)

In [ ]:
# Phi = ...  # design matrix (N, D)
# model = BayesianLinearRegression(Phi, y_train)
# model.compute_posterior(alpha=1.0, beta=25.0)

# Phi_star = ...  # test design matrix
# mean_f, var_f = model.predict_f(Phi_star)
# mean_y, var_y = model.predict_y(Phi_star)   # includes noise 1/beta

# alpha_opt, beta_opt = model.optimize_hyperparameters(alpha0=1.0, beta0=1.0)
# log_ml = model.compute_marginal_likelihood(alpha_opt, beta_opt)
# print(f'Optimised alpha={alpha_opt:.4f}, beta={beta_opt:.4f}, log_ml={log_ml:.4f}')

---
# Week 4 — Laplace Approximation & Classification

MAP: w* = argmax log p(w|y)  |  Laplace: q(w) = N(w*, (-∇² log p(w*|y))^{-1})  |  Probit: σ(μ_f / √(1 + πσ²_f/8))

In [ ]:
# model   = LogisticRegressionWithPrior(X_train, y_train, alpha=1.0)
# laplace = LaplaceApproximation(model.log_joint, D=X_train.shape[1])
# laplace.construct_approximation(w_init=np.zeros(X_train.shape[1]))
# print('MAP:', laplace.posterior_mean)

# ppd = PosteriorPredictiveDistribution(model, laplace)
# p_probit = ppd.probit_approx(X_test)
# p_mc     = ppd.montecarlo(X_test, num_samples=1000, seed=0)
# print(f'Accuracy: {accuracy(y_test, p_probit):.4f}')
# print(f'ELPD:     {elpd(y_test, p_probit):.4f}')

---
# Week 5 — Gaussian Process Regression

m*(x*) = k*^T (K+σ²I)^{-1} y  |  Var(f*) = k(x*,x*) - k*^T (K+σ²I)^{-1} k*

In [ ]:
# h = Hyperparameters(kappa=1.0, lengthscale=1.0, sigma=0.1)
# kernel = StationaryIsotropicKernel(squared_exponential)  # or matern12, matern32
# gp = GaussianProcessRegression(X_train, y_train, kernel)
# gp.set_hyperparameters(h)

# mean_f, var_f = gp.predict_f(X_test)
# mean_y, var_y = gp.predict_y(X_test)
# log_ml = gp.log_marginal_likelihood(h)

# h_opt = optimize_marginal_likelihood(gp, X_train, y_train, h)

# fig, ax = plt.subplots()
# plot_gp(ax, X_test.ravel(), mean_f, var_f)

---
# Week 6 — GP Classification

p(y=1|f)=σ(f)  |  Posterior ≈ N(f̂, (K^{-1}+W)^{-1})  |  Predictive: probit approx

In [ ]:
# h = Hyperparameters(kappa=1.0, lengthscale=1.0, sigma=0.0)
# kernel = StationaryIsotropicKernel(squared_exponential)
# gpc = GaussianProcessClassification(X_train, y_train, kernel, h)
# gpc.construct_laplace_approximation()

# mu_f, var_f = gpc.predict_f(X_test)
# p_y1 = gpc.predict_y(X_test)           # P(y=1|x*)
# # or: p_y1 = probit_approx_binary(mu_f, var_f)

---
# Week 7 — Multi-class Classification & Decision Theory

softmax(f)_k = exp(f_k)/Σ exp(f_j)  |  H[p] = -Σ p_k log p_k  |  E[U] = p^T U

In [ ]:
# model = BayesianLinearSoftmax(X_train, y_train, alpha=1.0, num_classes=K)
# p_y = model.predict_y(X_test, num_samples=500, seed=0)  # (N*, K)

# entropy    = compute_entropy(p_y)     # (N*,)
# confidence = compute_confidence(p_y)  # (N*,)

# U = np.eye(K)   # 0-1 utility (= argmax)
# eu = compute_expected_utility(U, p_y)
# optimal_action = np.argmax(eu, axis=1)

---
# Week 8 — Variational Inference (Analytical / Optimiser tools)

ELBO: L(λ) = E_q[log p(y,w)] - E_q[log q(w)]  |  Adam update

In [ ]:
# One-call Laplace
# w_map, cov = laplace_approximation(model.log_joint, w_init=np.zeros(D))

# Adam optimiser for ELBO
# opt = AdamOptimizer(lr=0.01)
# for i in range(2000):
#     g = grad_elbo(lam)  # your gradient function
#     lam = opt.step(lam, g)

# Synthetic data
# X, y = create_linear_regression_data(N=100, sigma=0.3, seed=0)

---
# Week 9 — Hamiltonian Monte Carlo

H(θ,ν) = -log p(θ|y) + ½ν^Tν  |  Leapfrog: ν -= η/2 ∇U; θ += η ν; ν -= η/2 ∇U  |  Accept: min(1, exp(-ΔH))

In [ ]:
# def log_target(theta):
#     return -0.5 * jnp.sum(theta**2)   # replace with your model

# samples = HMC(
#     log_target=log_target, num_iterations=2000,
#     theta0=jnp.zeros(D), num_leapfrog_steps=20, step_size=0.1, seed=0
# )  # shape: (2001, D)

# chains = samples[None]   # (1, 2001, D)
# print('R-hat:', compute_Rhat(chains))
# print('ESS:',   compute_effective_sample_size(chains))

---
# Week 10 — Variational GMM

q(Z,π,μ,Λ) factorised  |  CAVI: update q(Z), q(π,μ,Λ) in turn  |  ELBO monitored for convergence

In [ ]:
# D = X_train.shape[1]
# gmm = VariationalGMM(
#     X=X_train, K=5,
#     alpha0=1e-3, beta0=1.0,
#     m0=np.zeros(D), W0=np.eye(D), nu0=float(D+1)
# )
# gmm.fit(X_train, max_iter=100)  # X must be passed to fit too

# r_nk     = gmm.compute_component_probs(X_test)    # (N*, K) responsibilities
# log_pred = gmm.evaluate_log_predictive(X_test)    # scalar log marginal

# fig, ax = plt.subplots()
# plot_gmm_contours(ax, gmm, X=X_train)


---
# Week 11 — Black-Box Variational Inference (BBVI)

q(w)=N(m,diag(v))  |  ELBO≈(1/S)Σ log p(y,w_s) + H[q]  |  w_s = m + √v ⊙ ε_s, ε~N(0,I)

In [ ]:
# bbvi = BlackBoxVariationalInference(
#     X=X_train, y=y_train,
#     log_likelihood_fn=log_lik_sigmoid,  # or log_lik_probit, log_lik_robust
#     alpha=1.0, D=X_train.shape[1]
# )
# lam      = bbvi.fit(seed=0, num_iterations=2000, S=10)
# w_samples= bbvi.generate_posterior_samples(lam, num_samples=500, seed=1)  # (500, D)

# p_hat = jnp.mean(jax.nn.sigmoid(X_test @ w_samples.T), axis=1)  # posterior predictive

---
# Week 12 — Bayesian Neural Networks

h_l = tanh(W_l h_{l-1} + b_l)  |  MAP: argmax log p(y|w) + log p(w)  |  Prior: p(w)=N(0,σ²_w I)

In [ ]:
# net = NeuralNetwork(
#     layer_sizes=[X_train.shape[1], 50, 50, 1],
#     X_train=X_train, y_train=y_train,
#     prior_var=1.0, noise_var=0.1, seed=0
# )
# params = net.optimize(num_iterations=500, lr=1e-3)
# mean_pred = net.predict(X_test)     # (N*, 1)

# fig, ax = plt.subplots()
# plot_nn_predictions(ax, X_test, mean_pred.ravel(), std=None)

---
# Quick Reference Table

| Week | Model | Inference | Key class / function |
|------|-------|-----------|---------------------|
| 1 | Beta-Binomial | Exact | `BetaDistribution`, `compute_posterior` |
| 2 | Logistic reg | Grid approx | `GridApproximation2D` |
| 3 | Bayesian lin reg | Exact (conjugate) | `BayesianLinearRegression` |
| 4 | Logistic reg | Laplace | `LaplaceApproximation`, `PosteriorPredictiveDistribution` |
| 5 | GP | Exact (regression) | `GaussianProcessRegression` |
| 6 | GP | Laplace on GP | `GaussianProcessClassification` |
| 7 | Multi-class linear | Laplace | `BayesianLinearSoftmax` |
| 8 | Any | VI tools | `AdamOptimizer`, `laplace_approximation` |
| 9 | Any | HMC | `HMC`, `compute_Rhat`, `compute_effective_sample_size` |
| 10 | GMM | Variational CAVI | `VariationalGMM` |
| 11 | Classification | BBVI | `BlackBoxVariationalInference` |
| 12 | Neural Network | MAP | `NeuralNetwork` |

---
# Scratch — Exam Solutions

In [ ]:
# Q1


In [ ]:
# Q2


In [ ]:
# Q3


In [ ]:
# Q4


In [ ]:
# Q5
